# DocuMind PDF-extraction benchmark on a Kaggle GPU

**Before running:** Settings → Accelerator **GPU T4 x2** (or P100) · **Internet: On** · add the dataset that holds the bundle (`documind-pdfbench-kaggle.zip`, uploaded as a private dataset) under *Input*.

Then **Save Version → Save & Run All (Commit)** so the run continues without the browser. Expect 3–4 hours. Outputs in `/kaggle/working`: `pdfbench_results.zip` (extractor outputs, scores, retrieval scores, GPU lock files, environment facts; rewritten after every extractor so a killed run still leaves results) and `pdfbench_run.log`. The repo copy lives on the scratch disk under `/tmp/documind`.

In [ ]:
import os

os.environ.update({
    # run order; the driver skips tesseract automatically when the image only offers Tesseract 4.x
    # (the laptop run measured Tesseract 5.5.0 alone; set PDFBENCH_ALLOW_OLD_TESSERACT=1 to force it)
    "PDFBENCH_EXTRACTORS": "pymupdf,pdfplumber,pymupdf4llm,easyocr,docling,marker_noocr,mineru,tesseract",
    "PDFBENCH_MAX_RSS_GB": "12",     # per-worker memory cap (the laptop run used 4)
    "PDFBENCH_TIMEOUT_MIN": "30",    # per-document time cap (same as the laptop run)
    "PDFBENCH_DOWNSTREAM": "1",      # retrieval stage (bge-large + reranker) on the GPU afterwards
    "PDFBENCH_CLEAN_ENVS": "1",      # delete each heavy virtualenv after its run to stay within disk
})

In [ ]:
import os, shutil, zipfile
from pathlib import Path

ROOT = Path("/tmp/documind")            # scratch disk: keeps the 140 MB repo copy out of the notebook's output
INPUT = Path("/kaggle/input")
os.environ["PDFBENCH_ROOT"] = str(ROOT)

def find_bundle():
    """Kaggle usually extracts uploaded zips; handle both an extracted tree and a raw zip."""
    for run_py in INPUT.rglob("run.py"):
        if run_py.parent.name == "pdf_extraction" and (run_py.parents[2] / "data" / "benchmark_pdfs").is_dir():
            return run_py.parents[2], None
    zips = sorted(INPUT.rglob("*.zip"))
    return None, (zips[0] if zips else None)

if (ROOT / "benchmarks" / "pdf_extraction" / "run.py").exists():
    print("reusing", ROOT, "(resume)")
else:
    tree, bundle_zip = find_bundle()
    if tree is not None:
        print("copying", tree); shutil.copytree(tree, ROOT)
    elif bundle_zip is not None:
        print("extracting", bundle_zip); ROOT.mkdir(parents=True); zipfile.ZipFile(bundle_zip).extractall(ROOT)
    else:
        raise SystemExit("bundle not found under /kaggle/input: add the dataset in the right-hand panel")

pdfs = sorted(p.name for p in (ROOT / "data" / "benchmark_pdfs").glob("*.pdf"))
print(len(pdfs), "PDFs:", ", ".join(pdfs))
assert len(pdfs) >= 27, "corpus incomplete"

In [ ]:
!cd /tmp/documind && bash benchmarks/pdf_extraction/kaggle/run_kaggle.sh 2>&1 | tee /kaggle/working/pdfbench_run.log

In [ ]:
import collections, glob, json, os

out = "/tmp/documind/benchmarks/pdf_extraction/out"
if not os.path.isdir(out):
    print("no extractor output was produced; last lines of the driver log:")
    print("".join(open("/kaggle/working/pdfbench_run.log").readlines()[-30:]))
else:
    for name in sorted(os.listdir(out)):
        runs = [json.load(open(p)) for p in glob.glob(f"{out}/{name}/*/run.json")]
        counts = collections.Counter(r["status"] for r in runs)
        pages = sum(r.get("pages_processed") or 0 for r in runs if r["status"] == "ok")
        secs = sum(r.get("seconds") or 0 for r in runs if r["status"] == "ok")
        print(f"{name:14s} {dict(counts)}  {pages} pages, {secs/max(pages,1):.2f} s/page")

z = "/kaggle/working/pdfbench_results.zip"
print("\nDownload:", z, f"({os.path.getsize(z)/1e6:.0f} MB)" if os.path.exists(z) else "(missing)")